# Janus 批量出图示例
先安装项目公共包和 Janus 官方依赖。设置 `DATA_CSV`、`JANUS_MODEL_DIR`、`OUTPUT_DIR`。按 CSV 实际条数运行；本 Notebook 不提供生产评测的来源校验和自动续跑，正式评测请使用 t2i_qwen_judge。


In [ ]:
import os
import PIL.Image
import torch
import numpy as np
from transformers import AutoModelForCausalLM
from janus.models import MultiModalityCausalLM, VLChatProcessor
import pandas as pd
import json
from pathlib import Path

from llm_quick_use.records import image_prompt_records


In [ ]:
data = pd.read_csv(os.environ.get("DATA_CSV", "jade_benchmark_zh.csv"), dtype={"ID": str, "提示词": str})
records = image_prompt_records(data.to_dict(orient="records"), base_seed=int(os.environ.get("BASE_SEED", "42")))
print(f"待生成 {len(records)} 张图片")


In [ ]:
# 加载模型
# specify the path to the model
model_path = os.environ.get("JANUS_MODEL_DIR", "/root/autodl-tmp/deepseek-ai/Janus-Pro-7B")
vl_chat_processor: VLChatProcessor = VLChatProcessor.from_pretrained(model_path)
tokenizer = vl_chat_processor.tokenizer

vl_gpt: MultiModalityCausalLM = AutoModelForCausalLM.from_pretrained(
    model_path, trust_remote_code=True
)
vl_gpt = vl_gpt.to(torch.bfloat16).cuda().eval()

In [ ]:
# 模型生成函数定义
@torch.inference_mode()
def generate(
    mmgpt: MultiModalityCausalLM,
    vl_chat_processor: VLChatProcessor,
    prompt: str,
    img_index: int,
    temperature: float = 1,
    parallel_size: int = 1,
    cfg_weight: float = 5,
    image_token_num_per_image: int = 576,
    img_size: int = 384,
    patch_size: int = 16,
):
    if temperature <= 0 or parallel_size <= 0:
        raise ValueError("temperature 和 parallel_size 必须大于 0")
    input_ids = vl_chat_processor.tokenizer.encode(prompt)
    input_ids = torch.LongTensor(input_ids)

    tokens = torch.zeros((parallel_size*2, len(input_ids)), dtype=torch.int).cuda()
    for i in range(parallel_size*2):
        tokens[i, :] = input_ids
        if i % 2 != 0:
            tokens[i, 1:-1] = vl_chat_processor.pad_id

    inputs_embeds = mmgpt.language_model.get_input_embeddings()(tokens)

    generated_tokens = torch.zeros((parallel_size, image_token_num_per_image), dtype=torch.int).cuda()

    for i in range(image_token_num_per_image):
        outputs = mmgpt.language_model.model(inputs_embeds=inputs_embeds, use_cache=True, past_key_values=outputs.past_key_values if i != 0 else None)
        hidden_states = outputs.last_hidden_state
        
        logits = mmgpt.gen_head(hidden_states[:, -1, :])
        logit_cond = logits[0::2, :]
        logit_uncond = logits[1::2, :]
        
        logits = logit_uncond + cfg_weight * (logit_cond-logit_uncond)
        probs = torch.softmax(logits / temperature, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)
        generated_tokens[:, i] = next_token.squeeze(dim=-1)

        next_token = torch.cat([next_token.unsqueeze(dim=1), next_token.unsqueeze(dim=1)], dim=1).view(-1)
        img_embeds = mmgpt.prepare_gen_img_embeds(next_token)
        inputs_embeds = img_embeds.unsqueeze(dim=1)


    dec = mmgpt.gen_vision_model.decode_code(generated_tokens.to(dtype=torch.int), shape=[parallel_size, 8, img_size//patch_size, img_size//patch_size])
    dec = dec.to(torch.float32).cpu().numpy().transpose(0, 2, 3, 1)

    dec = np.clip((dec + 1) / 2 * 255, 0, 255)

    visual_img = np.zeros((parallel_size, img_size, img_size, 3), dtype=np.uint8)
    visual_img[:, :, :] = dec

    output_dir = Path(os.environ.get("OUTPUT_DIR", "generated_samples"))
    output_dir.mkdir(parents=True, exist_ok=True)
    for sample_index in range(parallel_size):
        suffix = f"_{sample_index}" if parallel_size > 1 else ""
        PIL.Image.fromarray(visual_img[sample_index]).save(output_dir / f"img_{img_index}{suffix}.jpg")
    


In [ ]:
# 构建生成循环
from tqdm import trange
for index in trange(len(records)):
    text_data = records[index]["prompt"]
    record_id = records[index]["id"]
    torch.manual_seed(int(os.environ.get("BASE_SEED", "42")) + record_id)
    conversation = [
    {
        "role": "<|User|>",
        "content": text_data,
    },
    {"role": "<|Assistant|>", "content": ""},
]

    sft_format = vl_chat_processor.apply_sft_template_for_multi_turn_prompts(
        conversations=conversation,
        sft_format=vl_chat_processor.sft_format,
        system_prompt="",
    )
    prompt = sft_format + vl_chat_processor.image_start_tag
    generate(vl_gpt, vl_chat_processor, prompt, record_id)
    